In [0]:
# Install Great Expectations
%pip install great-expectations==0.17.23
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Import Libraries
import great_expectations as ge
from pyspark.sql.functions import col, lit, when
from pyspark.sql.window import Window

In [0]:
# Read Data from Bronze Delta Lake
aircrafts_bronze_path = "s3://travel-analytics-bronze/delta/bronze/aircrafts/"
df_aircrafts = spark.read.format("delta").load(aircrafts_bronze_path)

print("=" * 80)
print("AIRCRAFT VALIDATION WITH GREAT EXPECTATIONS")
print("=" * 80)
print(f"Total records: {df_aircrafts.count()}")

AIRCRAFT VALIDATION WITH GREAT EXPECTATIONS
Total records: 492


In [0]:
ge_df = ge.from_pandas(df_aircrafts.toPandas())

print("\nRunning Great Expectations validation...")


Running Great Expectations validation...


In [0]:
# Define and Run Expectations

# Expectation 1: aircraft_id NOT NULL
result1 = ge_df.expect_column_values_to_not_be_null(column="aircraft_id")
print(f"✓ aircraft_id NOT NULL: {result1['success']}")

# Expectation 2: aircraft_id UNIQUE
result2 = ge_df.expect_column_values_to_be_unique(column="aircraft_id")
print(f"✓ aircraft_id UNIQUE: {result2['success']}")

# Expectation 3: aircraft_name NOT NULL
result3 = ge_df.expect_column_values_to_not_be_null(column="aircraft_name")
print(f"✓ aircraft_name NOT NULL: {result3['success']}")

✓ aircraft_id NOT NULL: True
✓ aircraft_id UNIQUE: False
✓ aircraft_name NOT NULL: True


In [0]:
#  Extract Failed Rows from Great Expectations Results

# Get unexpected values from each validation
failed_aircraft_id_null = set()
failed_aircraft_id_duplicate = set()
failed_aircraft_name_null = set()

# Extract failed rows for aircraft_id NULL
if not result1['success']:
    # Get row indices where aircraft_id is null
    df_temp = df_aircrafts.withColumn("row_id", monotonically_increasing_id())
    null_rows = df_temp.filter(col("aircraft_id").isNull())
    failed_aircraft_id_null = set([row['row_id'] for row in null_rows.collect()])

# Extract failed rows for aircraft_id NOT UNIQUE
if not result2['success']:
    from pyspark.sql.functions import count
    window_spec = Window.partitionBy("aircraft_id")
    df_temp = df_aircrafts.withColumn("row_id", monotonically_increasing_id())
    df_temp = df_temp.withColumn("count", count("aircraft_id").over(window_spec))
    duplicate_rows = df_temp.filter(col("count") > 1)
    failed_aircraft_id_duplicate = set([row['row_id'] for row in duplicate_rows.collect()])

# Extract failed rows for aircraft_name NULL
if not result3['success']:
    df_temp = df_aircrafts.withColumn("row_id", monotonically_increasing_id())
    null_name_rows = df_temp.filter(col("aircraft_name").isNull())
    failed_aircraft_name_null = set([row['row_id'] for row in null_name_rows.collect()])

# Combine all failed row IDs
all_failed_rows = failed_aircraft_id_null | failed_aircraft_id_duplicate | failed_aircraft_name_null


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8161356203081501>, line 19
     17 from pyspark.sql.functions import count
     18 window_spec = Window.partitionBy("aircraft_id")
---> 19 df_temp = df_aircrafts.withColumn("row_id", monotonically_increasing_id())
     20 df_temp = df_temp.withColumn("count", count("aircraft_id").over(window_spec))
     21 duplicate_rows = df_temp.filter(col("count") > 1)

NameError: name 'monotonically_increasing_id' is not defined

In [0]:
# STEP 6: Split Valid and Invalid Records

if len(all_failed_rows) > 0:
    # Add row_id to original dataframe
    df_with_id = df_aircrafts.withColumn("row_id", monotonically_increasing_id())
    
    # Split into valid and invalid
    df_invalid = df_with_id.filter(col("row_id").isin(list(all_failed_rows))).drop("row_id")
    df_valid = df_with_id.filter(~col("row_id").isin(list(all_failed_rows))).drop("row_id")
else:
    df_valid = df_aircrafts
    df_invalid = spark.createDataFrame([], df_aircrafts.schema)

valid_count = df_valid.count()
invalid_count = df_invalid.count()
total_count = df_aircrafts.count()

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)
print(f"✅ Valid records:   {valid_count} ({valid_count/total_count*100:.2f}%)")
print(f"❌ Invalid records: {invalid_count} ({invalid_count/total_count*100:.2f}%)")


In [0]:
#  Write Invalid Records to Quarantine

if invalid_count > 0:
    quarantine_path = "s3://travel-analytics-bronze/Quarantine/Aircrafts"
    
    df_invalid.write \
        .format("parquet") \
        .mode("append") \
        .save(quarantine_path)
    
    print(f"\n❌ Invalid records sent to Quarantine: {quarantine_path}")
    print("\n--- Sample Invalid Records ---")
    df_invalid.show(10, truncate=False)
else:
    print("\n✅ All records passed validation! No quarantine needed.")

print("\n" + "=" * 80)
print("✅ VALIDATION COMPLETED!")
print("=" * 80)
print(f"Valid records remain in Bronze: {aircrafts_bronze_path}")
print(f"Invalid records in Quarantine: s3://travel-analytics-bronze/Quarantine/Aircrafts")